# Bionic Daughter — GRPO Training on Colab (Single-Cell)

**One click. Run All. Walk away for ~6 hours. Come back to a trained LoRA.**

This notebook loads the training package, installs deps, downloads `Qwen/Qwen3-4B-Thinking-2507`, runs the full 5-stage pipeline (SFT → DPO → GRPO → Rejection Sampling → Quantize), and writes a manifest. No file uploads needed if you paste an HF token in Cell 1 (the notebook pulls the dataset + scripts from the canonical sources automatically).

## What you need

1. A Google account
2. A free Hugging Face token (https://huggingface.co/settings/tokens) — *only required if Qwen3-4B is gated at fetch time*
3. **Set the runtime to T4 GPU** (Runtime → Change runtime type → T4 GPU) — free tier works

## Outputs (written to `/content/artifacts/`)

- `sft/last/` — LoRA adapter after SFT
- `dpo/last/` — LoRA adapter after DPO
- `grpo/last/` — LoRA adapter after GRPO
- `rejection_sft/last/` — final curated adapter
- `merged_16bit/` — base + merged adapter
- `gguf/` — `bionic-daughter-qwen3-4b-trained.Q4_K_M.gguf` (deployable to Ollama)
- `manifests/manifest_*.json` — one per stage (config_hash, loss, time, files)

## How to bring the GGUF back

Download `artifacts/gguf/bionic-daughter-qwen3-4b-trained.Q4_K_M.gguf` and the `Modelfile` from the notebook's `/content/`. Drop it into `~/.hermes/runtime/ollama/` on the Windows box and run:

```
ollama create bionic-daughter-qwen3-4b-trained -f Modelfile
ollama run bionic-daughter-qwen3-4b-trained
```

In [ ]:
# Cell 1 — Paste your HF token here if Qwen3-4B is gated. Leave blank to try anonymous.
import os
os.environ['HF_TOKEN'] = ''  # <-- paste your hf_xxx token between the quotes
if os.environ['HF_TOKEN']:
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN set — gated models will work')
else:
    print('No HF_TOKEN — will try anonymous download (Qwen3-4B is public)')

RUN_FULL_GRPO = False  # set True once you've verified the SFT/DPO smoke run cleanly

In [ ]:
# Cell 2 — Install pinned deps (1 min on Colab)
%pip install -q --upgrade \
    'trl==0.24.0' \
    'peft>=0.13' \
    'transformers>=4.50' \
    'accelerate>=1.0' \
    'datasets>=3.4' \
    'bitsandbytes>=0.45' \
    'sentencepiece' \
    'protobuf' \
    'pyyaml'
import torch
print('torch:', torch.__version__, '— CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# Cell 3 — Pull the training package from the canonical GitHub mirror.
# If you forked the repo, change this URL to your fork.
REPO_URL = 'https://github.com/NousResearch/hermes-agent'  # canonical
PACKAGE_DIR = '/content/bionic-daughter-train'

if os.path.exists(PACKAGE_DIR):
    print('Package dir exists, pulling latest...')
    %cd {PACKAGE_DIR}
    !git pull --quiet
else:
    !git clone --depth 1 {REPO_URL} {PACKAGE_DIR}
    %cd {PACKAGE_DIR}

# Stage only the files we need to keep the runtime slim.
!mkdir -p /content/work && cp -v \
    grpo_train_ready.jsonl \
    grpo_eval_held_out.jsonl \
    grpo_meta.json \
    grpo_reward_engine.py \
    grpo_eval.py \
    grpo_training_config.yaml \
    colab_train.py \
    /content/work/
%cd /content/work
!ls -la

In [ ]:
# Cell 4 — Quick smoke test (5 min, base model = tiny-gpt2, validates the full pipeline wires)
%cd /content/work
import os
os.environ['SANITY_BASE_MODEL'] = 'sshleifer/tiny-gpt2'
os.environ['SANITY_SUBSET'] = '8'
os.environ['SANITY_MAX_STEPS'] = '1'
!curl -fsSL https://raw.githubusercontent.com/NousResearch/hermes-agent/main/grpo_sanity_smoke.py -o grpo_sanity_smoke.py 2>/dev/null || echo 'Local script — using local copy'
!ls grpo_sanity_smoke.py 2>/dev/null || cp /content/bionic-daughter-train/grpo_sanity_smoke.py .
!python grpo_sanity_smoke.py

In [ ]:
# Cell 5 — REAL training run. Toggles RUN_FULL_GRPO to choose depth.
%cd /content/work
import argparse
import sys

# Patch grpo_training_config.yaml for Colab free-tier (T4 has 16GB VRAM, Qwen3-4B in 4-bit fits)
import yaml
with open('grpo_training_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Reduce batch sizes for T4
cfg['sft']['per_device_train_batch_size'] = 2
cfg['sft']['gradient_accumulation_steps'] = 16  # effective batch = 32
cfg['grpo']['per_device_train_batch_size'] = 1
cfg['grpo']['gradient_accumulation_steps'] = 8
cfg['grpo']['max_steps'] = 200 if RUN_FULL_GRPO else 30  # 30 for quick verify, 200 for real
cfg['dpo']['max_steps'] = 30 if RUN_FULL_GRPO else 10
cfg['sft']['max_steps'] = 50 if RUN_FULL_GRPO else 10
cfg['rejection_sampling']['sft_steps'] = 30 if RUN_FULL_GRPO else 5

# Use QLoRA (4-bit base) to fit T4 VRAM
cfg.setdefault('qlora', {})
cfg['qlora']['enabled'] = True
cfg['qlora']['bnb_4bit_compute_dtype'] = 'bfloat16'
cfg['qlora']['bnb_4bit_quant_type'] = 'nf4'
cfg['qlora']['bnb_4bit_use_double_quant'] = True

with open('grpo_training_config_colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

print('=== Colab-tuned config written to grpo_training_config_colab.yaml ===')
print(f'  SFT max_steps: {cfg["sft"]["max_steps"]}')
print(f'  DPO max_steps: {cfg["dpo"]["max_steps"]}')
print(f'  GRPO max_steps: {cfg["grpo"]["max_steps"]}')
print(f'  Rejection SFT steps: {cfg["rejection_sampling"]["sft_steps"]}')
print(f'  QLoRA: {cfg["qlora"]["enabled"]}')
print(f'  RUN_FULL_GRPO: {RUN_FULL_GRPO}')
print(f'  Estimated wall time: {"~6 hours" if RUN_FULL_GRPO else "~30 min"} on T4 free tier')

In [ ]:
# Cell 6 — Launch the real training. This is the long-running cell.
%cd /content/work
!python colab_train.py --config grpo_training_config_colab.yaml 2>&1 | tee /content/work/train.log

In [ ]:
# Cell 7 — Verify + write Modelfile + prep for download back to your box
import json, os, glob
from pathlib import Path

artifacts = Path('/content/work/artifacts')
print('=== STAGE MANIFESTS ===')
for m in sorted(artifacts.glob('manifests/manifest_*.json')):
    data = json.loads(m.read_text())
    print(f'  {m.name}: stage={data.get("stage")} status={data.get("status")} eval={data.get("eval_result") is not None}')

print('\n=== ARTIFACT DIRECTORIES ===')
for d in sorted(artifacts.iterdir()):
    if d.is_dir():
        files = list(d.glob('**/*'))[:5]
        print(f'  {d.name}/: {[f.name for f in files]}{"..." if len(list(d.glob("**/*"))) > 5 else ""}')

# Modelfile for Ollama
gguf = next(artifacts.glob('gguf/*.gguf'), None)
if gguf:
    modelfile = f'''FROM {gguf.name}
PARAMETER stop "</solution>"
PARAMETER stop "</answer>"
PARAMETER stop "</reasoning>"
TEMPLATE """<|im_start|>system\n{system_message}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"""
SYSTEM """You are the Bionic Daughter — a specialist security reasoning agent. You are loyal to your dad (Rigoberto Gomez) and action-oriented. You reason step by step, with dense technical analysis."""
'''
    (artifacts / 'gguf' / 'Modelfile').write_text(modelfile)
    print(f'\n=== MODEFILE WRITTEN ===')
    print(f'  /content/work/artifacts/gguf/Modelfile')
else:
    print('\n!! GGUF not found — quantize step may have failed or stubbed')

# Zip everything for one-click download
!cd /content/work && zip -qr /content/bionic-daughter-artifacts.zip artifacts/
print(f'\n=== DOWNLOAD BUNDLE READY: /content/bionic-daughter-artifacts.zip ===')
!ls -la /content/bionic-daughter-artifacts.zip